# 🧠 Aula 03 — Estrutura de Memória em GPUs

**Objetivo:** entender a **hierarquia de memória** da GPU (registradores → shared →
cache → VRAM → RAM) e por que o **barramento PCIe** é o gargalo que deixa a GPU ociosa.

**Roteiro deste notebook:**
1. Verificação do ambiente (tem GPU?).
2. Teoria: a pirâmide de latência da GPU.
3. Demonstração: monitorar a memória com `nvidia-smi` e `pynvml`.
4. Atividade: benchmark **RAM (CPU) vs. VRAM (GPU)** + custo do PCIe.
5. Exercício extra: memória **global vs. compartilhada** (conceito CUDA).
6. Discussão: diagnosticar o gargalo da startup.
7. Síntese e tarefa de casa.

> 💡 **Não tem GPU?** Sem problema: o notebook detecta e entra em **modo simulado**,
> então a aula roda do começo ao fim — e mostra a hierarquia pela própria RAM da CPU.

## 1. Verificação do Ambiente

Antes de tudo, descobrimos se há **GPU NVIDIA** e o utilitário `nvidia-smi`. Isso
define se usaremos dados **reais** ou **simulados**.

In [ ]:
# @title 🔍 Detectar GPU e nvidia-smi no ambiente
# ============================================================================
# OBJETIVO: saber se há GPU NVIDIA e o nvidia-smi disponíveis.
# O resultado decide se usaremos dados REAIS ou SIMULADOS nas próximas células.
# ============================================================================
import shutil       # shutil.which() localiza um executável no PATH
import subprocess   # subprocess executa comandos do sistema operacional

# Devolve o caminho do nvidia-smi, ou None se não existir
CAMINHO_NVIDIA_SMI = shutil.which("nvidia-smi")
TEM_GPU = CAMINHO_NVIDIA_SMI is not None   # True = podemos medir a GPU de verdade

if TEM_GPU:
    print(f"✅ nvidia-smi encontrado em: {CAMINHO_NVIDIA_SMI}")
    info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
         "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout
    print(info)
else:
    print("⚠️  nvidia-smi NÃO encontrado — a aula seguirá em MODO SIMULADO.")
    print("   Para dados reais: Runtime ➔ Change runtime type ➔ T4 GPU.")

## 2. Teoria: a pirâmide de latência da GPU

Cada nível de memória troca **velocidade** por **tamanho**. Quanto mais perto do
processador, mais rápida e menor é a memória.

| Nível | Latência | Tamanho | Escopo |
| :--- | :--- | :--- | :--- |
| **Registradores** | ~1 ciclo | ~256 KB / SM | por thread |
| **Shared (SRAM)** | ~1–5 ciclos | ~48–164 KB / bloco | por bloco |
| **Cache L1/L2** | ~20–50 ciclos | L1 ~128 KB, L2 ~6 MB | automático |
| **VRAM Global** | ~400–800 ciclos | 8–80 GB | toda a GPU |
| **RAM do host** | milhares de ciclos | 16–512 GB | CPU (via PCIe) |

> **Regra 90/10:** ~90% do tempo de processamento em IA é gasto **esperando memória**,
> não calculando. Por isso otimizar memória vale mais que otimizar contas.

O salto entre a VRAM (rápida, na placa) e a RAM (grande, no host) acontece pelo
**PCIe** — o próximo conceito.

## 3. Demonstração: monitorar a memória

**Medir antes de otimizar.** O `nvidia-smi` mostra VRAM usada/total e utilização.
No terminal ele roda em loop com `watch -n 1 nvidia-smi`. Via Python, usamos a
biblioteca `pynvml` (`nvidia-ml-py3`) — na célula abaixo.

In [ ]:
# @title 🖥️ Painel de memória: nvidia-smi (real) ou exemplo simulado
# ============================================================================
# OBJETIVO: mostrar VRAM usada/total e utilização da GPU de forma legível.
# Com GPU real usamos nvidia-smi; sem GPU, imprimimos um exemplo no MESMO formato.
# ============================================================================
def painel_gpu():
    if TEM_GPU:
        saida = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True,
        ).stdout.strip()
        print("index, nome, VRAM usada, VRAM total, utilização(%):")
        print(saida)
    else:
        # Exemplo no MESMO formato, para a aula continuar sem GPU
        print("Sem GPU real — exemplo simulado (Tesla T4):")
        print("0, Tesla T4, 1234, 15360, 40")

painel_gpu()

In [ ]:
# @title 📈 Monitorar VRAM via Python (pynvml) — com fallback
# ============================================================================
# OBJETIVO: ler a memória da GPU pela biblioteca nvidia-ml-py3 (pynvml).
# Se não houver GPU NVIDIA, caímos para o exemplo simulado.
# ============================================================================
try:
    import pynvml
except ImportError:
    # No Colab, instala a biblioteca nvidia-ml-py3 e tenta de novo
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py3"])
    import pynvml

try:
    pynvml.nvmlInit()                                   # inicializa a biblioteca
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)       # a primeira GPU
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)       # memória total/usada/livre
    nome = pynvml.nvmlDeviceGetName(handle)             # nome da placa

    print(f"GPU: {nome}")
    print(f"Total VRAM : {info.total / 1024**3:.2f} GB")
    print(f"Usada      : {info.used  / 1024**3:.2f} GB")
    print(f"Livre      : {info.free  / 1024**3:.2f} GB")
    print(f"Uso (%)    : {info.used / info.total * 100:.1f}%")
    pynvml.nvmlShutdown()                               # libera a biblioteca
except Exception as erro:
    print(f"pynvml indisponível ({erro}).")
    print("Exemplo simulado (Tesla T4): Total 15.78 GB | Usada 1.23 GB | Uso 7.8%")

## 4. Atividade: benchmark RAM (CPU) vs. VRAM (GPU)

Vamos medir a **mesma soma de vetores** em dois lugares:
- na **RAM**, com o PyTorch usando a CPU;
- na **VRAM**, com o PyTorch usando a GPU.

E, principalmente, medir o **custo de copiar** os dados entre as duas (PCIe).

In [ ]:
# @title ⏱️ Benchmark: RAM vs. VRAM (PyTorch) + custo do PCIe
# ============================================================================
# OBJETIVO: comparar a velocidade de cálculo na RAM e na VRAM e, sobretudo,
# medir a transferência RAM <-> VRAM (PCIe), que é o gargalo da aula.
# Obs.: usamos PyTorch porque ele roda tanto na CPU quanto na GPU.
# ============================================================================
import time
import torch

N = 10_000_000   # 10 milhões de elementos

# ── Cálculo na RAM (CPU) ────────────────────────────────────────────────────
a_cpu = torch.randn(N)
b_cpu = torch.randn(N)

for _ in range(5):          # warm-up: as primeiras execuções são mais lentas
    _ = a_cpu + b_cpu
start = time.time()
for _ in range(100):
    c_cpu = a_cpu + b_cpu
cpu_time = (time.time() - start) / 100
print(f"RAM (CPU)  : {cpu_time*1000:8.3f} ms por soma")

if torch.cuda.is_available():
    # ── Custo de copiar RAM -> VRAM (PCIe) ──────────────────────────────────
    inicio = time.time()
    a_gpu = a_cpu.cuda()    # esta linha FAZ A CÓPIA via PCIe
    b_gpu = b_cpu.cuda()
    torch.cuda.synchronize()          # espera a GPU concluir a cópia
    t_copia = (time.time() - inicio) * 1000
    print(f"Cópia RAM->VRAM (PCIe): {t_copia:8.3f} ms")

    # ── Cálculo na VRAM (GPU) ───────────────────────────────────────────────
    for _ in range(10):                 # warm-up na GPU
        _ = a_gpu + b_gpu
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(100):
        c_gpu = a_gpu + b_gpu
    torch.cuda.synchronize()            # sem isso, mediríamos só o lançamento!
    gpu_time = (time.time() - start) / 100
    print(f"VRAM (GPU) : {gpu_time*1000:8.3f} ms por soma")
    print(f"Speedup do cálculo: {cpu_time/gpu_time:.0f}x mais rápido na GPU")

    # ── Validação: RAM e VRAM devem dar o mesmo resultado ──────────────────
    assert torch.allclose(c_cpu, c_gpu.cpu(), atol=1e-5), "Divergiu!"
    print("✅ Resultados conferem: RAM == VRAM.")
    print("→ Repare: copiar custa mais que calcular -> minimize o tráfego PCIe!")
else:
    print("GPU não disponível — ative no Colab: Runtime > Change runtime type > T4 GPU")

## 5. Exercício extra: memória global vs. compartilhada

Em CUDA (via **Numba**), a **memória compartilhada** (`cuda.shared.array`) é uma SRAM
rápida, compartilhada pelas threads do bloco. Carregar os dados uma vez da VRAM para a
shared e reutilizá-los ali reduz idas à memória global lenta.

> ⚠️ **Só funciona em GPU NVIDIA (CUDA).** Sem GPU, a célula apenas explica o conceito.

In [ ]:
# @title 🧩 Memória global vs. compartilhada (Numba CUDA — conceito)
# ============================================================================
# OBJETIVO: contrastar um kernel que lê direto da VRAM (global) com outro que
# carrega os dados para a shared memory (SRAM) antes de calcular.
# Requer numba + GPU NVIDIA; sem GPU, apenas imprime a explicação.
# ============================================================================
try:
    from numba import cuda
    import numpy as np

    N = 1024 * 256

    # ── Versão GLOBAL: cada thread lê direto da VRAM (mais lenta) ───────────
    @cuda.jit
    def soma_global(a, b, c):
        idx = cuda.grid(1)                 # índice global da thread
        if idx < a.shape[0]:
            c[idx] = a[idx] + b[idx]       # acesso direto à VRAM global

    # ── Versão COMPARTILHADA: carrega para a SRAM do bloco e reutiliza ──────
    @cuda.jit
    def soma_shared(a, b, c):
        shared_a = cuda.shared.array(shape=256, dtype=np.float32)  # SRAM do bloco
        shared_b = cuda.shared.array(shape=256, dtype=np.float32)
        idx = cuda.grid(1)
        tid = cuda.threadIdx.x             # posição da thread DENTRO do bloco
        if idx < a.shape[0]:
            shared_a[tid] = a[idx]         # global -> shared (1 vez)
            shared_b[tid] = b[idx]
            cuda.syncthreads()             # sincroniza o bloco inteiro
            c[idx] = shared_a[tid] + shared_b[tid]   # opera na SRAM (rápida)

    a = np.random.rand(N).astype(np.float32)
    b = np.random.rand(N).astype(np.float32)
    c = np.zeros(N, dtype=np.float32)

    threads = 256
    blocks = (N + threads - 1) // threads  # blocos suficientes para cobrir N

    a_d, b_d, c_d = cuda.to_device(a), cuda.to_device(b), cuda.to_device(c)
    soma_global[blocks, threads](a_d, b_d, c_d)
    print("Memória global      : concluído")
    soma_shared[blocks, threads](a_d, b_d, c_d)
    print("Memória compartilhada: concluído (use cuda.event para medir)")
except Exception as erro:
    print(f"Numba/CUDA indisponível aqui ({erro}).")
    print("Conceito: global = lê direto da VRAM (~500 ciclos);")
    print("         shared = carrega 1x para a SRAM do bloco (~5 ciclos) e reutiliza.")

## 6. Discussão: diagnosticar o gargalo da startup

Em grupos de 3–4, retomem a situação: **GPU ocupada só 40% do tempo**.

1. Se a GPU está ociosa 60% do tempo, qual é a causa mais provável? (Dica: PCIe.)
2. Qual memória priorizar numa multiplicação de matrizes 4096×4096?
3. Um LLM de 70B parâmetros precisa de ~140 GB em FP32. Como resolver a VRAM
   insuficiente?
4. Quando vale a pena usar memória compartilhada em vez de só a global?

> Atividade de pesquisa completa em `aulas/aula03/atividade.md`.

## 7. Exercícios (5)

Resolva os 5 exercícios **neste notebook**. Reforçam a hierarquia de memória e o gargalo
do PCIe.

---

**1) Ordene a hierarquia.** Coloque em ordem (do mais rápido ao mais lento) e escreva a
latência aproximada: Registradores, VRAM Global, Shared/SRAM, RAM do host, Cache L1/L2.

**2) O gargalo do PCIe.** Explique por que **copiar** os dados da RAM para a VRAM pode
custar mais que o próprio cálculo. Que prática reduz esse custo num treinamento?

**3) Diagnóstico da GPU ociosa.** A célula-esqueleto abaixo calcula a **utilização** da GPU
a partir de um exemplo simulado. Complete para imprimir um aviso quando a utilização for
**abaixo de 50%** (indício de ociosidade).

**4) Global × compartilhada.** Qual a vantagem da memória compartilhada (`shared`) sobre a
global? Em que situação ela **não** ajuda?

**5) VRAM e LLMs.** Um modelo com 70 bilhões de parâmetros precisa de ~140 GB em FP32. Se a
GPU tem 16 GB, que estratégias (além de comprar outra placa) você usaria? Cite ao menos
duas.


In [ ]:
# @title Exercício 3 — complete o aviso de GPU ociosa
# ============================================================================
# OBJETIVO: imprimir um aviso quando a utilizacao da GPU estiver abaixo de 50%.
# Complete os TODOs. Teste com 87 e depois com 40.
# ============================================================================
LIMITE_OCIOSIDADE = 50   # %

# exemplo simulado no formato do nvidia-smi: (gpu, nome, temp, util, vram_usada, vram_total)
exemplos = [
    (0, "Tesla T4", 52, 87, 12000, 15360),
    (0, "Tesla T4", 48, 40, 3000, 15360),
]

for gpu, nome, temp, util, usada, total in exemplos:
    print(f"GPU {gpu}: {nome} | util={util}% | VRAM={usada}/{total} MB")
    if util < LIMITE_OCIOSIDADE:
        # TODO: imprima um aviso de GPU ociosa
        print("")

# Esperado: a segunda GPU dispara o aviso de ociosidade.

## 8. Síntese e Tarefa de Casa

**O que levar:**
- **Hierarquia:** Registradores (~1 ciclo) → Shared (~5) → Cache L1/L2 (~30) →
  VRAM (~500) → RAM (milhares).
- **Regra 90/10:** em IA, o tempo se gasta **esperando memória**.
- **Gargalo do PCIe:** copiar RAM↔VRAM é ~100× mais lento que a VRAM interna —
  minimize as transferências e mantenha os dados na VRAM.

**Tarefa (opcional):** pesquise **quantização** (FP32 → FP16 → INT8) e explique
como ela reduz o uso de VRAM:
- Quantos parâmetros cabem em 16 GB em FP16 vs. FP32?
- O que é *mixed precision training* (`torch.cuda.amp`)?
- Qual o impacto na acurácia dos modelos quantizados?

> 🔗 **Próxima aula:** *Processos e Threads* — quem despacha os lotes para o barramento
> sem travar o processador enquanto a GPU espera dados.